
<h3 align="center">SA4110 MACHINE LEARNING APPLICATION DEVELOPMENT</h3>
<h4 align="center">CA - GROUP 3 - IMAGE CLASSIFIER</h4>
<hr>

<div class="alert alert-block alert-info">
<b><u>Tasks:</u></b>
<ol>
<li>Create an Image Classifier (CNN model) to classify images of fruits correctly.</li>
<li>A Fruits Dataset is provided that consists of these 3 Classes: -</li>
    <ul>
    <li>Apple</li>
    <li>Banana</li>
    <li>Orange</li>
    </ul>
<li>Use the Images in train.zip and test.zip to Train and Test your Image Classifier.</li>
<li>Document your experiments and results in improving your model’s accuracy.</li>
<li>The Following Activities can Improve your Model’s Accuracy: -</li>
    <ul>
    <li>Balance out the Number of Samples in Each Class</li>
    <li>Correct any mis-labelling in any of the 3 Classes</li>
    <li>Image Augmentation to Generate more Data </li>
    </ul>
<li>Use Matplotlib to Generate any Plots that can Help the Reader understand your Work Better.</li>
</ol></div>



## Set Up
1. Installations
2. Imports

In [23]:
# ========================================================== #
# 1. (Optional) Verification of Base Libraries / Packages
# Purpose: Verify that the host environment has the necessary
# packages or libraries required to run the script.
# ========================================================== #

import sys
import subprocess
import importlib.util

def verify_and_install():
    required_packages = {
        'pandas': 'pandas',
        'numpy': 'numpy',
        'matplotlib': 'matplotlib',
        'seaborn': 'seaborn',
        'IPython': 'ipython',
        'PIL': 'Pillow',
        'torch': 'torch',
        'torchvision': 'torchvision',
        'keras': 'keras',
        'sklearn': 'scikit-learn'
    }

    print("--- Verifying Environment Setup ---")

    for import_name, pip_name in required_packages.items():
        # Check if the module can be found in the host environment
        if importlib.util.find_spec(import_name) is None:
            print(f"❌ '{import_name}' is missing. Installing '{pip_name}'...")
            try:
                # Run pip install securely via the current python executable
                subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
                print(f"✅ Successfully installed '{pip_name}'.")
            except subprocess.CalledProcessError as e:
                # Stop the script if a critical package fails to install
                print(f"⚠️ Failed to install '{pip_name}'. Error: {e}")
                sys.exit(1)
        else:
            print(f"✅ '{import_name}' is already installed.")

    print("-" * 70)
    print("All necessary packages are ready. Proceeding with the script...")
    print("-" * 70 + "\n")

# Run the verification before importing the third-party libraries
verify_and_install()

--- Verifying Environment Setup ---
✅ 'pandas' is already installed.
✅ 'numpy' is already installed.
✅ 'matplotlib' is already installed.
✅ 'seaborn' is already installed.
✅ 'IPython' is already installed.
✅ 'PIL' is already installed.
✅ 'torch' is already installed.
✅ 'torchvision' is already installed.
✅ 'keras' is already installed.
✅ 'sklearn' is already installed.
----------------------------------------------------------------------
All necessary packages are ready. Proceeding with the script...
----------------------------------------------------------------------



In [24]:
# ========================================================== #
# 2. Importing the Required Libraries and Packages
# Purpose: Import the Libraries and Packages to the Host
# Environment for Data Manipulation, Visualization, Image Processing,
# Deep Learning, and Machine Learning Utilities.
# ========================================================== #

# A. System and OS-level operations
import os
import statistics
from pathlib import Path
os.environ["KERAS_BACKEND"] = "torch"

# B. Data Manipulation and Mathematical Operations
import pandas as pd
import numpy as np

# C. Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# D. Image Processing
from PIL import Image

# E. Deep Learning Frameworks (PyTorch & Keras)
import torch
import keras
from keras import backend as K
from keras.models import Sequential
from keras.layers import (Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout)
from keras.optimizers import Adam
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

# F. Machine Learning Utilities and Metrics (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# G. Time reporting
import time

# Clear session and random seeds
import random

K.clear_session()

def set_seeds(seed=42):
    # 1. Python
    random.seed(seed)

    # 2. NumPy operations
    np.random.seed(seed)

    # 3. Core PyTorch CPU operations
    torch.manual_seed(seed)

    # Keras setting
    keras.utils.set_random_seed(seed)

    print(f"✅ All random seeds locked globally to: {seed}")

# Execute the function
set_seeds(42)

✅ All random seeds locked globally to: 42


## Image Preprocessing
1. Load image from Test and Train folders
2. Preprocess and label images
3. Data preparation

In [25]:
# ========================================================== #
# 3. Helper functions to render HTML components for display
# ========================================================== #
# Render dataframe as HTML table
def render_df(df, sample_size):

    # Dynamic sampling based on input size
    display_df = (
        df if sample_size == 0
        else df.sample(n=min(sample_size, len(df)))
    )

    return (display_df.style
        .set_properties(**{'text-align': 'center', 'color': 'black'})
        .set_table_styles([
            {'selector':'tr:nth-child(even)', 'props': [('background-color', 'azure')]},
            {'selector':'th', 'props':[('text-align', 'center'), ('color', 'black'), ('font-weight', 'bold')]},
            {'selector':'tr:nth-child(odd)', 'props':[('text-align', 'center'), ('background-color', 'paleturquoise')]}
        ])).to_html()

# Render section header as HTML line
def render_section_header(title):
    return f"<h4 style='text-align: left; margin-top: 15px; margin-bottom: 5px;'>{title}</h4>"

# Label value table
def render_key_value_table(rows):
    """
    rows = [
        ("Label 1", "Value 1"),
        ("Label 2", "Value 2"),
    ]
    """

    html = "<table style='margin: 0; text-align: left; border: none;'>"

    for i, (label, value) in enumerate(rows):
        bg = "paleturquoise" if i % 2 == 0 else "azure"

        html += f"""
        <tr style='background-color: {bg};'>
            <td style='color: black;
                       text-align: center;
                       padding-right: 10px;
                       border: none;'>
                <strong>{label}</strong>
            </td>
            <td style='color: black;
                       text-align: center;
                       border: none;'>
                {value}
            </td>
        </tr>
        """

    html += "</table>"
    return html

# Status messages
def render_status_list(messages):
    """
    messages = [
        ("green", "Success"),
        ("red", "Failure")
    ]
    """

    html = "<ul style='text-align: left;'>"

    for color, msg in messages:
        html += f"<li style='color: {color};'>{msg}</li>"

    html += "</ul>"
    return html

# Line separator
def render_separator():
    return "<hr style='margin-top: 15px; border-top: 1px solid #ddd;'>"

# Not found message
def render_no_images_message():
    return "<div style='color: red; text-align: center;'>No valid .jpg images found.</div>"

# Display dataset inspect
def render_dataset_info(train_logs, train_html, test_logs, test_html):
    return f"""
    <div style="display: flex; flex-direction: row; justify-content: space-around; align-items: flex-start; gap: 20px;">
        <div style="flex: 1; padding: 10px; border: 1px solid #ccc; border-radius: 16px;">
            <h3 style="text-align: center; color: 'black';">Training & Validation Dataset</h3>
            <div style="display: flex; flex-direction: column; justify-content: center;">
                {train_logs}
            </div>
            <h4 style="text-align: left;">DataFrame Sample</h4>
            <div style="display: flex; flex-direction: column; justify-content: center;">
                {train_html}
            </div>
        </div>

        <div style="flex: 1; padding: 10px; border: 1px solid #ccc; border-radius: 16px;">
            <h3 style="text-align: center; color: 'black';">Testing Dataset</h3>
            <div style="display: flex; flex-direction: column; justify-content: center;">
                {test_logs}
            </div>
            <h4 style="text-align: left;">DataFrame Sample</h4>
            <div style="display: flex; flex-direction: column; justify-content: center;">
                {test_html}
            </div>
        </div>
    </div>
    """

# Display data set info
def render_dataset_shapes(train_shape_html, val_shape_html, class_test_shape_html):
    return f"""
    <div style="display: flex; flex-direction: row; justify-content: space-around; align-items: flex-start; gap: 20px;">
        <div style="flex: 1; padding: 10px; border: 1px solid #ccc; border-radius: 16px;">
            <h3 style="text-align: center; color: 'black';">Training (80%)<br>Dataset Shape</h3>
            <div style="display: flex; flex-direction: column; justify-content: center;">
                {train_shape_html}
            </div>
        </div>

        <div style="flex: 1; padding: 10px; border: 1px solid #ccc; border-radius: 16px;">
            <h3 style="text-align: center; color: 'black';">Validation (20%)<br>Dataset Shape</h3>
            <div style="display: flex; flex-direction: column; justify-content: center;">
                {val_shape_html}
            </div>
        </div>

        <div style="flex: 1; padding: 10px; border: 1px solid #ccc; border-radius: 16px;">
            <h3 style="text-align: center; color: 'black';">Testing<br>Dataset Shape</h3>
            <div style="display: flex; flex-direction: column; justify-content: center;">
                {class_test_shape_html}
            </div>
        </div>
    </div>
    """

In [26]:
# ========================================================== #
# 4. Load and Process the Image Data from the Directories
# Purpose: Scan the 'train' and 'test' directories for .jpg
# images, extract class labels from filenames, calculate
# median dimensions, and compile the data into DataFrames
# for prior for model training and evaluation.
# ========================================================== #

def process_image_directory(directory_path, dataset_name="Dataset"):

    file_names = []
    class_labels = []
    widths = []
    heights = []
    status_messages = []

    # Single pass through the directory
    for img_path in directory_path.glob("*.jpg"):
        try:
            # A. Open a valid data image file and extract its dimensions
            with Image.open(img_path) as img:
                widths.append(img.size[0])
                heights.append(img.size[1])

            # B. Extract metadata ONLY if the image opened successfully
            file_names.append(img_path.name)
            class_labels.append(img_path.name.strip().split('_')[0])

        except Exception as e:
            # C. Exception handling for invalid data image file
            status_messages.append(("red", f"Error reading {img_path.name}: {e}"))

    # Success message if no errors
    if not status_messages:
        status_messages.append(("green", "All images read successfully without errors."))

    # Build the DataFrame Table of Consisting of File Name and its Classes
    df = pd.DataFrame({
        'fileName': file_names,
        'classLabel': class_labels
    })

    # Build HTML Report

    log_html = ""

    log_html += render_section_header("Processing Status")
    log_html += render_status_list(status_messages)

    log_html += render_section_header("Class Labels Found")
    log_html += render_key_value_table([
        (f"Class {i}", label)
        for i, label in enumerate(df['classLabel'].unique(), start = 1)
    ])

    # Calculate Medians of the data images
    median_w, median_h = 0, 0
    log_html += render_section_header("<br>Dataset Statistics")

    if widths and heights:
        median_w = statistics.median(widths)
        median_h = statistics.median(heights)

        log_html += render_key_value_table([
            ("Total Images Evaluated", len(widths)),
            ("Image Median Width", f"{median_w} px"),
            ("Image Median Height", f"{median_h} px")
        ])
    else:
        log_html += render_no_images_message()

    log_html += render_separator()

    return df, median_w, median_h, log_html

In [27]:
# ---------------------------------------------------------- #
# Execute the Function for Both Directories
# ---------------------------------------------------------- #

# Define the path directories
base_dir = Path.cwd().parents[0]
img_train_dir = base_dir / "train"
img_test_dir = base_dir / "test"

# Process Training and Validation Datasets from 'train' directory
class_train_df, img_T_widths_median, img_T_heights_median, train_logs = process_image_directory(
    directory_path = img_train_dir,
    dataset_name = "Training and Validation Dataset"
)

# Sampling of the training and validation dataframe for verification
train_html = render_df(class_train_df, 5)

# Process Testing Datasets from 'test' directory
class_test_df, img_t_widths_median, img_t_heights_median, test_logs = process_image_directory(
    directory_path = img_test_dir,
    dataset_name = "Testing Dataset"
)

# Sampling of the testing dataframe for verification
test_html = render_df(class_test_df, 5)

# Wrapping them html in a Flex container to enable side-by-side display
train_test_html = render_dataset_info(train_logs, train_html, test_logs, test_html)
display(HTML(train_test_html))

,fileName,classLabel
,fileName,classLabel


In [28]:
# ========================================================== #
# 5. Define a New Class: FruitLabeller and Supporting Functions
# Purpose: Create a custom PyTorch Dataset to load images, apply
# transformations, and encode string labels to numeric indices
# using a provided DataFrame. Includes a helper function to
# visually summarize the dataset's class distribution.
# ========================================================== #

class FruitLabeller(Dataset):
    def __init__(self, class_dataframe, img_directory, transforms = None):
        self.dataframe = class_dataframe
        self.directory = img_directory
        self.transforms = transforms

        unique_labels = sorted(class_dataframe['classLabel'].unique())
        self.label_map = {label: idx for idx, label in enumerate(unique_labels)}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx]['fileName']
        img_path = self.directory / img_name
        image = Image.open(img_path).convert("RGB")

        label_str = self.dataframe.iloc[idx]['classLabel']
        label = self.label_map[label_str]

        if self.transforms:
            image = self.transforms(image)

        return image, label

def render_dataset_shape(df):
    shape_df = df['classLabel'].value_counts().sort_index().reset_index()
    shape_df.columns = ['Class Label', 'Total Image']
    return shape_df

In [29]:
# ========================================================== #
# 6. Data preparation
# ========================================================== #

# Set Target Size (Max 224x224)
target_width = int(img_T_widths_median) if img_T_widths_median < 224 else 224
target_height = int(img_T_heights_median) if img_T_heights_median < 224 else 224

# Split Data (80% Train, 20% Val)
train_df, val_df = train_test_split(
    class_train_df,
    test_size=0.2,
    random_state=42,
    stratify=class_train_df['classLabel']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# Define PyTorch Transformations
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop((target_height, target_width)),
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop((target_height, target_width)),
    transforms.ToTensor()
])

# Instantiate PyTorch Datasets
train_dataset = FruitLabeller(train_df, img_train_dir, train_transforms)
val_dataset = FruitLabeller(val_df, img_train_dir, test_transforms)
test_dataset = FruitLabeller(class_test_df, img_test_dir, test_transforms)

# Create PyTorch DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Display Dataset Shapes
train_shape_html = render_df(render_dataset_shape(train_df), 0)
val_shape_html = render_df(render_dataset_shape(val_df), 0)
class_test_shape_html = render_df(render_dataset_shape(class_test_df), 0)

# Wrapping them html in a Flex container to enable side-by-side display
dataset_shapes_html = render_dataset_shapes(train_shape_html, val_shape_html, class_test_shape_html)

display(HTML(dataset_shapes_html))

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
# Display Sample Images
images, labels = next(iter(train_loader))
class_names = {v: k for k, v in train_dataset.label_map.items()}

# Set up a MatPlotLib Grid
fig, axes = plt.subplots(3, 3, figsize=(8, 8))

for i, ax in enumerate(axes.flat):
    img = np.transpose(images[i].numpy(), (1, 2, 0))
    img = np.clip(img, 0, 1)
    ax.imshow(img)

    # Look up the Integer Label in the Dictionary and Set it as the Title
    label_idx = labels[i].item()
    ax.set_title(class_names[label_idx].capitalize(), fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Experiment 1: Baseline Model
1. Build CNN architecture
2. Compile model
3. Train model
4. Evaluate model

In [ ]:
# ========================================================== #
# 7. Build and Compile CNN Architecture (224 x 224 x 3)
# Model is coded to be configurable for experimentation
# By default no dropout or augmentation
# ========================================================== #
def build_model(dropout_rate = 0.0, data_augmentation = None):
    model_layers = []

    # Only add augmentation if passed in
    if data_augmentation:
        model_layers.extend(data_augmentation)

    # Default settings
    model_layers.extend([
        Input(shape=(3, 224, 224)),

        # Conv1: 32 filters, Output shape: 224 x 224 x 32
        # Scans the raw image using a kernel to detect basic, low-level spatial features
        Conv2D(
            filters=32,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
            data_format="channels_first" # PyTorch default
        ),

        # MaxPool: 224 x 224 -> 112 x 112
        # Downsamples feature maps with the maximum value from a 2 x 2 window
        # Reduces computational cost, controls overfitting, and makes the network
        # robust to small shifts or translations in the image.
        MaxPooling2D(pool_size=(2, 2), data_format="channels_first"),

        # Conv2: 64 filters, output shape: 112 x 112 x 64
        # Scans pooled feature maps again with an increased num of filters (64) to learn
        # higher-level, more complex and abstract features
        Conv2D(
            filters=64,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
            data_format="channels_first"
        ),

        # MaxPool: 112 x 112 -> 56 x 56
        MaxPooling2D(pool_size=(2, 2), data_format="channels_first"),


        # Conv3: 128 filters, output shape: 56 x 56 x 128
        # Scans pooled feature maps again with an increased num of filters (128) to learn
        # higher-level, more complex and abstract features
        Conv2D(
            filters=128,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
            data_format="channels_first"
        ),

        # MaxPool: 56 x 56 -> 28 x 28
        MaxPooling2D(pool_size=(2, 2), data_format="channels_first"),

        # Dropout layer
        Dropout(dropout_rate),

        # Flatten: 28 x 28 x 128 = 100,352
        # Flattens conv layers (3D tensors) into a 1D vector as required by Dense layers
        Flatten(),

        # Fully connected layer
        # learns correlations between combined features and specific fruit classes
        Dense(128, activation="relu"),

        # Output Layer: 3 nodes (Apples, Bananas, Oranges)
        # PURPOSE: Produces the final prediction.
        # - activation ('softmax'): Converts the raw output scores into a probability
        # distribution, where all outputs sum up to 1. The highest probability is the prediction.
        Dense(3, activation='softmax')
    ])

    return Sequential(model_layers)

# Build model
baseline_model = build_model()

In [ ]:
# ========================================================== #
# 8. Compile model
# Optimizer: Adam - Update weights while learning
# Loss: Sparse categorical crossentropy - Numerical labels
# Metrics: Accuracy - Measure correct / total preds
# ========================================================== #
def compile_model(model, learning_rate = 0.001):
    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

compile_model(baseline_model, 0.001)

# Model summary
baseline_model.summary()

In [ ]:
# ========================================================== #
# 9. Train CNN Model
# Epochs at 20 to allow liberal and fair experimentation
# Track processing time for experiment comparison
# ========================================================== #

# To store training/inference times for experiment comparison
processing_times = {}

def train_model(model_type, model, epoc):
    start = time.perf_counter()

    history = model.fit(
        train_loader,
        validation_data = val_loader,
        epochs = epoc
    )

    end = time.perf_counter()
    training_time = end - start

    processing_times[(model_type.lower() + '_training_time')] = training_time # Store time in dict

    return history

baseline_history = train_model('baseline', baseline_model, 20)

In [ ]:
# Helpers to prepare for model evaluation
# Configured to be reusable for every experiment

# Get predictions for model
# Track processing time for experiment comparison
def get_predictions(model_type, model):

    start = time.perf_counter()

    # Initialize lists to store true and predicted labels for the entire dataset
    y_true = []
    y_pred = []

    # Iterate through the entire test_loader
    for images, labels in test_loader:
        # Get predictions for the current batch (verbose=0 suppresses per-batch output)
        batch_preds_prob = model.predict(images, verbose=0)
        batch_preds = np.argmax(batch_preds_prob, axis=1)

        # Append the results to the tracking lists
        y_true.extend(labels.numpy())
        y_pred.extend(batch_preds)

    end = time.perf_counter()
    inference_time = end - start
    processing_times[(model_type.lower() + '_inference_time')] = inference_time # Store time in dict

    return y_true, y_pred

# Set target names
target_names = [
    name for name in train_dataset.label_map.keys()
]

# Evaluation result builder
def build_result(
    model_name,
    history,
    y_true,
    y_pred,
):
    report = classification_report(
        y_true,
        y_pred,
        target_names = target_names,
        output_dict = True
    )

    return {
        'name': model_name,
        'history': history,

        # For experiment comparison
        'accuracy': report['accuracy'],
        'precision': report['macro avg']['precision'],
        'recall': report['macro avg']['recall'],
        'f1': report['macro avg']['f1-score'],

        # For individual reporting
        'classification_report': pd.DataFrame(report)
            .transpose()
            .round(3),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

# Generate and render accuracy and loss graphs
def render_accuracy_loss_report(model_name, history, test_loss, test_accuracy):

    # Extract Key Metrics from the History object
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    sns.set_theme(style='darkgrid')
    plt.figure(figsize=(12, 5))

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy', marker='o', color='blue')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='o', color='darkorange')
    plt.title('Training and Validation Accuracy', fontsize=12, fontweight='bold')
    plt.xlabel('Epochs', fontsize=10, fontweight='bold')
    plt.ylabel('Accuracy', fontsize=10, fontweight='bold')
    plt.legend(loc='lower right')
    plt.xlim(0,21)
    plt.ylim(0,1.1)
    plt.grid(True)

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss', marker='o', color='blue')
    plt.plot(epochs_range, val_loss, label='Validation Loss', marker='o', color='darkorange')
    plt.title('Training and Validation Loss', fontsize=12, fontweight='bold')
    plt.xlabel('Epochs', fontsize=10, fontweight='bold')
    plt.ylabel('Loss', fontsize=10, fontweight='bold')
    plt.legend(loc='upper right')
    plt.xlim(0,21)
    plt.grid(True)

    plt.tight_layout()

    # Save figure as image
    filename = (f"{model_name.lower().replace(' ', '_')}_history.png")
    plt.savefig(filename)
    plt.close()

    # Return as HTML report
    return (
        render_section_header(
            f"Accuracy and Loss: {model_name}"
        )
        + f"""
        <p><b>Final Test Loss:</b> {test_loss:.4f}<br><b>Final Test Accuracy:</b> {test_accuracy:.4f}</p>

        <img src="{filename}"
             width="800">
        """
    )

# Generate and render accuracy and loss graphs
def render_accuracy_loss_report_epoch30(model_name, history, test_loss, test_accuracy):

    # Extract Key Metrics from the History object
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    sns.set_theme(style='darkgrid')
    plt.figure(figsize=(12, 5))

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy', marker='o', color='blue')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='o', color='darkorange')
    plt.title('Training and Validation Accuracy', fontsize=12, fontweight='bold')
    plt.xlabel('Epochs', fontsize=10, fontweight='bold')
    plt.ylabel('Accuracy', fontsize=10, fontweight='bold')
    plt.legend(loc='lower right')
    plt.xlim(0,31)
    plt.ylim(0,1.1)
    plt.grid(True)

    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss', marker='o', color='blue')
    plt.plot(epochs_range, val_loss, label='Validation Loss', marker='o', color='darkorange')
    plt.title('Training and Validation Loss', fontsize=12, fontweight='bold')
    plt.xlabel('Epochs', fontsize=10, fontweight='bold')
    plt.ylabel('Loss', fontsize=10, fontweight='bold')
    plt.legend(loc='upper right')
    plt.xlim(0,31)
    plt.grid(True)

    plt.tight_layout()

    # Save figure as image
    filename = (f"{model_name.lower().replace(' ', '_')}_history.png")
    plt.savefig(filename)
    plt.close()

    # Return as HTML report
    return (
        render_section_header(
            f"Accuracy and Loss: {model_name}"
        )
        + f"""
        <p><b>Final Test Loss:</b> {test_loss:.4f}<br><b>Final Test Accuracy:</b> {test_accuracy:.4f}</p>

        <img src="{filename}"
             width="800">
        """
    )

# Render classification report
def render_classification_report(result):
    html = render_section_header(f"Classification Report: {result['name']}")
    html += render_df(result['classification_report'], 0)
    return html

# Render confusion matrix
def render_confusion_matrix(result):
    cm = result['confusion_matrix']
    model_name = result['name']

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names,
                yticklabels=target_names)
    # plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')

    # Save figure as image
    filename = (f"{model_name.lower().replace(' ', '_')}_cm.png")
    plt.savefig(filename)
    plt.close()

    # Return as HTML report
    return (
        render_section_header(
            f"Confusion Matrix: {model_name}"
        )
        + f"""
        <img src="{filename}"
             width="800">
        """
    )

# Render Processing time
def render_processing_time(result, training_time, inference_time):
    model_name = result['name']
    return (
        render_section_header(
            f"Processing Time: {model_name}"
        )
        + f"""
            <p><b>Training Time:</b> {training_time:.4f}s<br><b>Inference Time:</b> {inference_time:.4f}s</p>
        """
    )

# Display all evaluation results
def display_evaluation_results(accuracy_loss_report, class_report, confusion_matrix, processing_time):
    display(HTML(accuracy_loss_report))
    display(HTML(class_report))
    display(HTML(confusion_matrix))
    display(HTML(processing_time))

In [ ]:
# ========================================================== #
# 10. Evaluate Baseline Model
# Configured to be reused on evaluation of experiments
# ========================================================== #

# Get predictions for model
y_true, y_pred, = get_predictions("baseline", baseline_model)

# Build evaluation result
baseline_result = build_result(
    "Baseline Model",
    baseline_history,
    y_true,
    y_pred,
)

# Generate and render Accuracy and Loss Report
test_loss, test_accuracy = baseline_model.evaluate(test_loader, verbose = 0)
baseline_accuracy_loss_report = render_accuracy_loss_report("Baseline Model", baseline_history, test_loss, test_accuracy)

# Generate and render Classification Report from result
baseline_class_report = render_classification_report(baseline_result)

# Generate and render Confusion Matrix
baseline_confusion_matrix = render_confusion_matrix(baseline_result)

# Render processing time
baseline_processing_time = render_processing_time(baseline_result, processing_times['baseline_training_time'], processing_times['baseline_inference_time'])

# Display evaluation results
display_evaluation_results(baseline_accuracy_loss_report, baseline_class_report, baseline_confusion_matrix, baseline_processing_time)

## Experiment 2: Improved Model with Hyperparameter Tuning
1. Rebuild CNN architecture
2. Compile model
3. Train model
4. Evaluate model

In [ ]:
# ========================================================== #
# 11. Compile and Train Model with Dropout at 0.4
# ========================================================== #

dropout_model_04 = build_model(0.4, False)
compile_model(dropout_model_04, 0.001)

# Model summary
dropout_model_04.summary()

dropout_history_04 = train_model('dropout04', dropout_model_04, 20)

In [ ]:
# ========================================================== #
# 12. Compile and Train Model with Dropout at 0.3
# ========================================================== #

dropout_model_03 = build_model(0.3, False)
compile_model(dropout_model_03, 0.001)

# Model summary
dropout_model_03.summary()

dropout_history_03 = train_model('dropout03', dropout_model_03, 20)

In [ ]:
# ========================================================== #
# 13. Evaluate Model with Dropout layer at 0.4
# ========================================================== #

# Get predictions for model
y_true, y_pred, = get_predictions("dropout04", dropout_model_04)

# Build evaluation result
dropout_result_04 = build_result(
    "Dropout Model 0.4",
    dropout_history_04,
    y_true,
    y_pred,
)

# Generate and render Accuracy and Loss Report
test_loss, test_accuracy = dropout_model_04.evaluate(test_loader, verbose = 0)
dropout_accuracy_loss_report_04 = render_accuracy_loss_report("Dropout Model 0.4", dropout_history_04, test_loss, test_accuracy)

# Generate and render Classification Report from result
dropout_class_report_04 = render_classification_report(dropout_result_04)

# Generate and render Confusion Matrix
dropout_confusion_matrix_04 = render_confusion_matrix(dropout_result_04)

# Render processing time
dropout_processing_time_04 = render_processing_time(dropout_result_04, processing_times['dropout04_training_time'], processing_times['dropout04_inference_time'])

# Display evaluation results
display_evaluation_results(dropout_accuracy_loss_report_04, dropout_class_report_04, dropout_confusion_matrix_04, dropout_processing_time_04)

In [ ]:
# ========================================================== #
# 14. Evaluate Model with Dropout layer at 0.3
# ========================================================== #

# Get predictions for model
y_true, y_pred, = get_predictions("dropout03", dropout_model_03)

# Build evaluation result
dropout_result_03 = build_result(
    "Dropout Model 0.3",
    dropout_history_03,
    y_true,
    y_pred,
)

# Generate and render Accuracy and Loss Report
test_loss, test_accuracy = dropout_model_03.evaluate(test_loader, verbose = 0)
dropout_accuracy_loss_report_03 = render_accuracy_loss_report("Dropout Model 0.3", dropout_history_03, test_loss, test_accuracy)

# Generate and render Classification Report from result
dropout_class_report_03 = render_classification_report(dropout_result_03)

# Generate and render Confusion Matrix
dropout_confusion_matrix_03 = render_confusion_matrix(dropout_result_03)

# Render processing time
dropout_processing_time_03 = render_processing_time(dropout_result_03, processing_times['dropout03_training_time'], processing_times['dropout03_inference_time'])

# Display evaluation results
display_evaluation_results(dropout_accuracy_loss_report_03, dropout_class_report_03, dropout_confusion_matrix_03, dropout_processing_time_03)

In [ ]:
# ====================================================================== #
# 15. Compile and Train Model with Learning Rate at 0.0005, Epochs = 30
# ====================================================================== #

LR_model_0005 = build_model(0.0, False)
compile_model(LR_model_0005, 0.0005)

# Model summary
LR_model_0005.summary()

LR_history_0005 = train_model('LR_0005', LR_model_0005, 30)

In [ ]:
# =============================================================== #
# 16. Evaluate Model with Learning Rate at 0.0005, Epochs = 30
# =============================================================== #

# Get predictions for model
y_true, y_pred, = get_predictions("LR_0005", LR_model_0005)

# Build evaluation result
LR_result_0005 = build_result(
    "Learning Rate 0.0005 Model",
    LR_history_0005,
    y_true,
    y_pred,
)

# Generate and render Accuracy and Loss Report
test_loss, test_accuracy = LR_model_0005.evaluate(test_loader, verbose = 0)
LR_accuracy_loss_report_0005 = render_accuracy_loss_report_epoch30("Learning Rate 0.0005 Model", LR_history_0005, test_loss, test_accuracy)

# Generate and render Classification Report from result
LR_class_report_0005 = render_classification_report(LR_result_0005)

# Generate and render Confusion Matrix
LR_confusion_matrix_0005 = render_confusion_matrix(LR_result_0005)

# Render processing time
LR_processing_time_0005 = render_processing_time(LR_result_0005, processing_times['lr_0005_training_time'], processing_times['lr_0005_inference_time'])

# Display evaluation results
display_evaluation_results(LR_accuracy_loss_report_0005, LR_class_report_0005, LR_confusion_matrix_0005, LR_processing_time_0005)

In [ ]:
# ====================================================================================== #
# 17. Compile and Train Model with Learning Rate at 0.0005 and Dropout at 0.3, Epochs 30
# ====================================================================================== #

LR_Dropout_model = build_model(0.3, False)
compile_model(LR_Dropout_model, 0.0005)

# Model summary
LR_Dropout_model.summary()

LR_Dropout_history = train_model('LR_Dropout', LR_Dropout_model, 30)

In [ ]:
# ============================================================================= #
# 18. Evaluate Model with Learning Rate at 0.0005 and Dropout at 0.3, Epochs 30
# ============================================================================= #

# Get predictions for model
y_true, y_pred, = get_predictions("LR_Dropout", LR_Dropout_model)

# Build evaluation result
LR_Dropout_result = build_result(
    "Learning Rate 0.0005 Dropout 0.3 Model",
    LR_Dropout_history,
    y_true,
    y_pred,
)

# Generate and render Accuracy and Loss Report
test_loss, test_accuracy = LR_Dropout_model.evaluate(test_loader, verbose = 0)
LR_Dropout_accuracy_loss_report = render_accuracy_loss_report_epoch30("Learning Rate 0.0005 Dropout 0.3 Model", LR_Dropout_history, test_loss, test_accuracy)

# Generate and render Classification Report from result
LR_Dropout_class_report = render_classification_report(LR_Dropout_result)

# Generate and render Confusion Matrix
LR_Dropout_confusion_matrix = render_confusion_matrix(LR_Dropout_result)

# Render processing time
LR_Dropout_processing_time = render_processing_time(LR_Dropout_result, processing_times['lr_dropout_training_time'], processing_times['lr_dropout_inference_time'])

# Display evaluation results
display_evaluation_results(LR_Dropout_accuracy_loss_report, LR_Dropout_class_report, LR_Dropout_confusion_matrix, LR_Dropout_processing_time)

## Results of Experiment 2
Note #1: All models with Learning Rate at 0.0005 have epochs = 30.
Other models have the same settings (including epochs = 20) as baseline model,
apart from the adjustments made to Dropout

1. Learning Rate 0.0005 model (BEST)
    - Final Test Loss: 0.3211
    - Final Test Accuracy: 0.9818
2. Learning Rate 0.0005 and Dropout 0.3 model
    - Final Test Loss: 0.3364
    - Final Test Accuracy: 0.9636
3. Dropout 0.3 model
    - Final Test Loss: 0.3650
    - Final Test Accuracy: 0.9636
4. Dropout 0.4 model
    - Final Test Loss: 0.4508
    - Final Test Accuracy: 0.9273

## Experiment 3: Data Augmentation Model
1. Rebuild CNN architecture
2. Compile model
3. Train model
4. Evaluate model

**Experiment 3A:** Geometric Modifications

In [ ]:
import keras
from keras import layers

# 1. Define the Data Augmentation Layers
exp3a_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
]

# 2. Instantiate the Experiment 3a Model
model_experiment_3a = build_model(dropout_rate=0.4, data_augmentation=exp3a_augmentation_layers)

In [ ]:
# 1. Clear memory from the previous run
keras.backend.clear_session()
set_seeds(42)

# 2. Recreate DataLoaders with your chosen optimal batch size
OPTIMAL_BATCH_SIZE = 32
temp_train_loader = DataLoader(train_dataset, batch_size=OPTIMAL_BATCH_SIZE, shuffle=True)
temp_val_loader = DataLoader(val_dataset, batch_size=OPTIMAL_BATCH_SIZE, shuffle=False)

# 3. Compile the model using custom helper function
# Sticking to the standard learning rate used in Exp 2 (0.001)
compile_model(model_experiment_3a, 0.001)

# 4. Print the summary to verify the augmentation layers are included at the top
model_experiment_3a.summary()

# 5. Train using custom training helper function
exp3a_history = train_model('augmentation_exp3a', model_experiment_3a, 20)

In [ ]:
# ========================================================
# # 13. Evaluate Model with Data Augmentation and Dropout
# ========================================================

# 1. Get predictions for the new model
y_true, y_pred = get_predictions("augmentation_exp3a", model_experiment_3a)

# 2. Build the result registry profile
exp3a_result = build_result(
    "Data Augmentation Model 0.4",
    exp3a_history,
    y_true,
    y_pred
)

# 3. Generate and render Accuracy and Loss Report
test_loss, test_accuracy = model_experiment_3a.evaluate(test_loader, verbose=0)
exp3a_accuracy_loss_report = render_accuracy_loss_report("Data Augmentation Model 0.4", exp3a_history, test_loss, test_accuracy)

# 4. Generate Classification Report and Confusion Matrix
exp3a_class_report = render_classification_report(exp3a_result)
exp3a_confusion_matrix = render_confusion_matrix(exp3a_result)

# 5. Render processing times
exp3a_processing_time = render_processing_time(
    exp3a_result,
    processing_times['augmentation_exp3a_training_time'],
    processing_times['augmentation_exp3a_inference_time']
)

# 6. Display the final evaluation dashboard
display_evaluation_results(exp3a_accuracy_loss_report, exp3a_class_report, exp3a_confusion_matrix, exp3a_processing_time)

**Experiment 3B**: Photometric Modifications

In [ ]:
# 1. Define the Data Augmentation Blueprint Layers
exp3b_augmentation_layers = [
    layers.RandomBrightness(factor=0.2),
    layers.RandomContrast(factor=0.2),
]

# 2. Instantiate the Experiment 3b Model
model_experiment_3b = build_model(dropout_rate=0.4, data_augmentation=exp3b_augmentation_layers)

In [ ]:
# 1. Clear memory from the previous run
keras.backend.clear_session()
set_seeds(42)

# 2. Recreate DataLoaders with your chosen optimal batch size
OPTIMAL_BATCH_SIZE = 32
temp_train_loader = DataLoader(train_dataset, batch_size=OPTIMAL_BATCH_SIZE, shuffle=True)
temp_val_loader = DataLoader(val_dataset, batch_size=OPTIMAL_BATCH_SIZE, shuffle=False)

# 3. Compile the model using custom helper function
# Sticking to the standard learning rate used in Exp 2 (0.001)
compile_model(model_experiment_3b, 0.001)

# 4. Print the summary to verify the augmentation layers are included at the top
model_experiment_3b.summary()

# 5. Train using custom training helper function
exp3b_history = train_model('augmentation_exp3b', model_experiment_3b, 20)

In [ ]:
# ========================================================
# # 13. Evaluate Model with Data Augmentation and Dropout
# ========================================================

# 1. Get predictions for the new model
y_true, y_pred = get_predictions("augmentation_exp3b", model_experiment_3b)

# 2. Build the result registry profile
exp3b_result = build_result(
    "Data Augmentation Model 0.4",
    exp3b_history,
    y_true,
    y_pred
)

# 3. Generate and render Accuracy and Loss Report
test_loss, test_accuracy = model_experiment_3b.evaluate(test_loader, verbose=0)
exp3b_accuracy_loss_report = render_accuracy_loss_report("Data Augmentation Model 0.4", exp3b_history, test_loss, test_accuracy)

# 4. Generate Classification Report and Confusion Matrix
exp3b_class_report = render_classification_report(exp3b_result)
exp3b_confusion_matrix = render_confusion_matrix(exp3b_result)

# 5. Render processing times
exp3b_processing_time = render_processing_time(
    exp3b_result,
    processing_times['augmentation_exp3b_training_time'],
    processing_times['augmentation_exp3b_inference_time']
)

# 6. Display the final evaluation dashboard
display_evaluation_results(exp3b_accuracy_loss_report, exp3b_class_report, exp3b_confusion_matrix, exp3b_processing_time)

## Experiment 4: Transfer Learning Model
1. Rebuild CNN architecture
2. Compile model
3. Train model
4. Evaluate model

## Summary of Findings
1. Misclassified Images
2. Test Accuracy across Experiments
3. Model Efficiency across Experiments